# 资源分配问题

In [30]:
import numpy as np
import random
import copy 
import time

## 类定义
定义两个类，node表示节点类， net表示线网类

In [31]:
class node():
    def __init__(self, ind=None, name=None, ziyuan=None, assignment=None):
        self.id = ind
        self.name = name
        self.ziyuan = ziyuan         # 节点所用资源的列表(在当前题目要求下，资源总和即可，不需要列表)
        self.assignment = assignment # 节点被分配到的FPGA序号
        self.net = []               # 节点所属线网的序号列表


class net():
    def __init__(self, nodelist=None):
        self.nodelist = nodelist        # 线网中的节点的列表，此列表中每个元素是节点的名字
        self.assignment = set()    # 线网中的节点被分配到的FPGA的集合， 此集合中每个元素是FPGA的序号


## 定义函数
### 文件信息读取
读取文件，取得节点资源信息

In [32]:
def readNodeFile(filename):
    f = open(filename)
    vertex = {}        # 字典
    vertexname=[]      # 列表
    ind = 0
    for line in f:
        list1 = line.split() 
        nm = list1[0]  #节点名
        vertexname.append(nm)  #将节点名加入列表便于后续查找
        ziyuan = list(map(int, list1[-10:]))  #得到需要的资源数列表 并在下面求和加入到 节点属性中
        vertex[nm] = node(ind, nm, sum(ziyuan)) 
        ind += 1
    f.close()
    return vertex,vertexname

读取文件，取得线网信息

In [33]:
def readNetFile(filename,vertex):
    f = open(filename)
    linestack=[]  #利用栈的方式来处理属于同一线网的节点 将其放在同一列表下
    for line in f:
        linestack.append(line.split())
    nets=[]
    nodelist=[]
    ind=0
    while linestack!=[]:
        line = linestack.pop()
        nodelist.append(line[0])
        if 's' in line:
            nets.append(net(nodelist)) #把每个线网的类 加入nets中 
            for node in nodelist:
                vertex[node].net.append(ind) #并给每个节点加上属于线网的序号
            nodelist=[]
            ind += 1
    f.close()
    return nets

### 定义写文件函数writeResult

In [34]:
def writeResult(F,vertexname):
    nosFPGA = len(F)
    for fpga in range(nosFPGA):
        ft.write("F"+str(fpga))
        for onenode in F[fpga]:
            ft.write(" "+vertexname[onenode])
        ft.write("\n")

In [35]:
# 产生N个染色体的初始群体,保存在pop里
# 每个染色体代表53个节点属于的染色体 输入染色体种群，染色体中节点数  节点属于的FPGA数目
def initPop(N,vertexname,nosFPGA):
    nosnode = len(vertexname)
    pop = np.zeros([N,nosnode],dtype=np.int8)    
    for i in range(N): #使用随机函数生成N个染色体            
        pop[i,:] =  [np.random.randint(0,nosFPGA) for j in range(nosnode) ] # 给53个节点每个加上 0 1 2 3的随机数 ,randint 随机产生a-b之间的整数，包括a和b。             
    return pop

In [36]:
# 根据种群 计算每个染色体中 FPGA分配资源的方差
def ziyuanCal(pop,nosFPGA,vertex,vertexname):
    N,nc=np.shape(pop)
    ziyuan_list = np.zeros([N,4])
    for i in range(N):
        gene=pop[i,:]
        ziyuan_list[i] = calStdoneGene(gene,nosFPGA,vertex,vertexname)
    return ziyuan_list

def calStdoneGene(gene,nosFPGA,vertex,vertexname):       
    F = [[] for i in range(nosFPGA)]
    Z=[]
    nc = len(gene)
    #  0 1 2 3 四个FPGA ，其中每个节点分别取这四个值 把对应值的资源数加到对应FPGA中
    for i in range(nc): 
        F[gene[i]].append(vertex[vertexname[i]].ziyuan)
    # 对FPGA中资源进行求和 再计算得到方差
    for i in range(nosFPGA):
        Z.append(sum(F[i]))
    return Z

# 根据种群 计算连线总和
def LinkCal(pop,nosFPGA,vertexname,nets):
    N,nc=np.shape(pop)
    Sum_link = np.zeros([N,1])
    for i in range(N):
        gene=pop[i,:]
        Sum_link[i] = linkCaloneGene(gene,nosFPGA,vertexname,nets)
    return Sum_link

def linkCaloneGene(gene,nosFPGA,vertexname,nets):
    L = [0 for i in range(nosFPGA)]
    nc = len(gene)
    #  0 1 2 3 四个FPGA ，其中每个节点分别取这四个值 把对应值的资源数加到对应FPGA中
    for i in range(nc):
        for onenet in nets:
            if vertexname[i] in onenet.nodelist:
                onenet.assignment.add(gene[i])   

    for onenet in nets:
        if len(onenet.assignment)==1:
            continue
        for fpgaInd in onenet.assignment:
            L[fpgaInd] += 1
        onenet.assignment = set()  # 计算完清空便于下一次计算
    sum_link = sum(L)
    return sum_link

In [37]:
# 根据所有染色体的方差，计算各个染色体的适应值fitness
def calFitness_1(ziyuan):
	i,j = np.shape(ziyuan)
	ziyuan_std=np.zeros([i,1])
	for i in range(i):
		ziyuan_std[i,0] = np.std(ziyuan[i,:])
	fitness = 1.0/ziyuan_std	
	return fitness

# 根据连线数，计算各个染色体的适应值fitness
def calFitness_2(Sum_link):
	fitness = 1.0/Sum_link	
	return fitness

#考虑到连线和资源数、计算各个染色体的适应值fitness  
def calFitness_3(Sum_link,ziyuan_list):
	i,j = np.shape(ziyuan_list)
	ziyuan_std=np.zeros([i,1])
	for i in range(i):
		ziyuan_std[i,0] = np.std(ziyuan_list[i,:])
	fitness = 10.0/ziyuan_std
	fitness = fitness/Sum_link
	return fitness


In [38]:
# 根据染色体群体pop已经对应的适应值fitness，
# 找到最高的适应值f，以及对应的染色体bst和其在pop中的编号/下标ind
def findBest(pop,fitness):
	f = np.max(fitness)
	ind = np.asarray(np.where(fitness == f)).flatten()
	bst = pop[ind,:]
	return [bst,f,ind]

def satified(fitness):
	return 0

In [39]:
# 根据染色体的适应值，按照一定的概率，生成新一代染色体群体newpop
def chooseNewP(pop,fitness):
	N,nc=np.shape(pop)
	fitness = np.cumsum(fitness)
	lst=np.zeros([N,1]) # 100个数
	rvalLst = np.random.rand(N,1)
	for i in range(N):
		rval = np.random.rand()
		lst[i]=0
		for j in range(N-1,-1,-1): # 生成N-1 到 0的序列 递减
			if rval>fitness[j]:
				lst[i]=j
				break				
	newpop = pop[list(lst.flatten().astype(np.uint8)),:]  
	return newpop

In [40]:
# 根据交叉概率pc，以及各染色体的适应值fitness，通过交叉的方式生成新群体
# #SelfAdj = 1时为自适应，否则取固定的交叉概率pc
def crossPop(pop,pc,fitness,SelfAdj):
	N,nc=np.shape(pop)
	lst=list(range(N))  # 用于随机操作一条染色体
	np.random.shuffle(lst)
	fm = np.max(fitness)
	fa = np.mean(fitness)
	k1 = pc
	k2 = pc
	i = 0
 
	while i < int(N/2):
		rval = np.random.rand() # 生成0-1的随机数
		j = np.random.randint(int(N/2),N)  # 区分有无np的
		if SelfAdj==1: 		# 自适应，参考149页
			fprime = np.max(fitness[i],fitness[j])
			if fprime>fa:
				pc = k1*(fm-fprime) / (fm-fa)
			else :
				pc = k2
		if rval > pc:  # 如果随机数大于交叉概率 则不进行交叉操作
			continue
		partner1 = copy.copy(pop[lst[i],:])
		partner2 = copy.copy(pop[lst[j],:])
		# if (partner1==partner2).all():
		# 	continue
		child1,child2 = genecross(partner1,partner2)
		pop[lst[i],:]=copy.copy(child1)
		pop[lst[j],:]=copy.copy(child2)
		i = i+1

In [41]:
# 父染色体partner1,partner2，通过交叉方式
# 生成两个子染色体child1,child2  通过单交叉点的方式
def genecross(partner1,partner2):
	length = len(partner1)
    #在种群个数内随机生成单点交叉点
	cpoint=random.randint(1,length-1)  # 不包括length
	# partner1 和 partner2 的后cpoint到length-1进行交换
	child1 = copy.copy(partner1)
	child2 = copy.copy(partner2)
	child1[cpoint:length-1] = partner2[cpoint:length-1]
	child2[cpoint:length-1] = partner1[cpoint:length-1] 
 
	return [child1,child2]

In [42]:
'''
函数作用：对种群进行变异操作，产生新染色体
输入参数：pop：种群  pw:变异概率   fitness:种群中每个染色体的适应值   SelfAdj：自适应
返回值:
'''
def mutPop(pop,pw,fitness,SelfAdj):  
	N,nc=np.shape(pop)	
	fm = max(fitness)
	fa = np.mean(fitness)
	k3 = pw
	k4 = pw
	for i in range(N):
		rval = random.random()
		if SelfAdj==1: 		# 自适应，参考149页
			if fitness[i]>fa:
				pw = k3*(fm-fitness[i]) / (fm-fa)
			else :
				pw = k4
		if rval > pw:  # 同上交叉操作，如果随机数大于概率就不进行变异操作，否者进行 
			continue
		pop[i,:] = np.asarray(mutZiyuanGene(pop[i,:])) # 将输入数据转为矩阵模式 

In [43]:
# 根据所有染色体的方差  对不符合条件的fitness设置为0
def Restraint(pop,fitness,nosFPGA,vertex,vertexname):
	pop_ziyuan = ziyuanCal(pop,nosFPGA,vertex,vertexname)
	for i in range(len(vertexname)):
		ziyuan_mean = np.mean(pop_ziyuan[i,:])
		for j in range(len(pop_ziyuan[i,:])):
			if pop_ziyuan[i,j]>1.1*ziyuan_mean:
				fitness[i,0]=0
			if pop_ziyuan[i,j]<0.9*ziyuan_mean:
				fitness[i,0]=0	
	return fitness

In [44]:
# 随机选取一个位置进行变异  可以预测到这种方法得到的结果不会很好
def mutZiyuanGene(gene): 
	nc = len(gene)
	pos1,pos2 = 0,0
	while pos1==pos2:
		pos1 = random.randint(0,nc-1)  # 随机选取某一个基因位置发生变异
		pos2 = random.randint(0,nc-1)  
	gene[pos1],gene[pos2] = gene[pos2],gene[pos1] #交换位置，完成变异
	return gene

### 清空分配信息
包括节点和线网中的分配信息

In [45]:
def clearAssignInfo(vertex,vertexname,nets):
    for nodeInd in range(len(vertexname)):
        vertex[vertexname[nodeInd]].assignment = None
    for onenet in nets:
        onenet.assignment = set()

## 执行代码
读取文件

In [46]:
# 读取节点文件
filename = "design.are"
vertex,vertexname = readNodeFile(filename)

In [47]:
# 读取线网文件
filename = "design.net"
nets = readNetFile(filename,vertex)

In [48]:
nosFPGA = 4 

问题一

In [49]:
'''
超参数设置：
'''
def GA_Fenpei_1(nosFPGA,vertex,vertexname):
	nc = len(vertexname)		# 节点个数
	N = 100		# 染色体群体规模
	MAXITER = 500	# 最大循环次数
	SelfAdj = 0 	#SelfAdj = 1时为自适应
	randAccordDist = 0	# 在mutate的时候根据位置值分配对应的概率，此概率决定哪个城市参与mutate

	pc = 0.50		# 交叉概率
	pw = 0.50		# 变异概率

	# 步骤1，产生N个染色体的初始群体,保存在pop里面
	pop = initPop(N,vertexname,nosFPGA)	#pop 遗传算法中的种群
	iter=0
	tstart=time.time()
	while iter<MAXITER:
		iter = iter+1
		ziyuan_list=ziyuanCal(pop,nosFPGA,vertex,vertexname);	#步骤2：计算每个染色体的路径长度
		
		fitness=calFitness_1(ziyuan_list)			#      计算每个染色体的适应值	
		b,f,ind=findBest(pop,fitness)			# 找到在当前种群中，适应度最高的个体

		# 步骤3：如果满足某些标准，算法停止 在此处没有设置条件
		if satified(fitness):
			break
		pop = chooseNewP(pop,fitness)				#      否则，选取出一个新的群体
		crossPop(pop,pc,fitness,SelfAdj)	#步骤4：交叉产生新染色体，得到新群体		
		mutPop(pop,pw,fitness,SelfAdj) 		#步骤5：基因变异	
		pop[0,:] = b[0,:]						# 保留上一代中适应值最高的染色体             

		Std_ziyuan = np.std(calStdoneGene(b[0,:],nosFPGA,vertex,vertexname))
		if np.mod(iter,MAXITER/10)==1:
			titer=time.time()
			print("iter = %d after running %d seconds with Std=%f" %(iter, int(titer-tstart),Std_ziyuan))

	#输出最优染色体/路径
	print('nc=%d,N=%d, MAXITER=%d, pc=%f, pw=%f ' %(nc,N, MAXITER,pc ,pw))
	print('最小方差为：%f,适应值为：%f,对应的分配方法为：' %(Std_ziyuan,f))
	print(b[0,:])

	#计算运行时间
	tend=time.time()
	print('The program need :  %f seconds.' %(tend-tstart))

	#计算得到F
	F=[[] for i in range(nosFPGA)]
	for i in range(nc): 
		F[b[0][i]].append(vertex[vertexname[i]].id)
	return Std_ziyuan,F

问题2

In [50]:
'''
超参数设置：
'''
def GA_Fenpei_2(nosFPGA,vertex,vertexname,nets):
	nc = len(vertexname)		# 节点个数
	N = 100		# 染色体群体规模
	MAXITER = 700	# 最大循环次数
	SelfAdj = 0 	#SelfAdj = 1时为自适应
	randAccordDist = 0	# 在mutate的时候根据位置值分配对应的概率，此概率决定哪个城市参与mutate

	pc = 0.75		# 交叉概率
	pw = 0.25		# 变异概率

	# 步骤1，产生N个染色体的初始群体,保存在pop里面
	pop = initPop(N,vertexname,nosFPGA)	#pop 遗传算法中的种群
	iter=0
	tstart=time.time()
	while iter<MAXITER:
		iter = iter+1
		Sum_link=LinkCal(pop,nosFPGA,vertexname,nets);	#步骤2：计算每个染色体的路径长度
		
		fitness=calFitness_2(Sum_link)			#      计算每个染色体的适应值	
		b,f,ind=findBest(pop,fitness)			# 找到在当前种群中，适应度最高的个体

		# 步骤3：如果满足某些标准，算法停止 在此处没有设置条件
		if satified(fitness):
			break
		pop = chooseNewP(pop,fitness)				#      否则，选取出一个新的群体
		crossPop(pop,pc,fitness,SelfAdj)	#步骤4：交叉产生新染色体，得到新群体		
		mutPop(pop,pw,fitness,SelfAdj) 		#步骤5：基因变异	
		pop[0,:] = b[0,:]						# 保留上一代中适应值最高的染色体             

		onesum_link = linkCaloneGene(b[0,:],nosFPGA,vertexname,nets)
		if np.mod(iter,MAXITER/10)==1:
			titer=time.time()
			print("iter = %d after running %d seconds with Sum=%f" %(iter, int(titer-tstart),onesum_link))

	#输出最优染色体/路径
	print('nc=%d,N=%d, MAXITER=%d, pc=%f, pw=%f ' %(nc,N, MAXITER,pc ,pw))
	print('最小连线为：%f,适应值为：%f,对应的分配方法为：' %(onesum_link,f))
	print(b[0,:])

	#计算运行时间
	tend=time.time()
	print('The program need :  %f seconds.' %(tend-tstart))

	#计算得到F
	F=[[] for i in range(nosFPGA)]
	for i in range(nc): 
		F[b[0][i]].append(vertex[vertexname[i]].id)
	return onesum_link,F

问题3

In [51]:
'''
超参数设置：
'''
def GA_Fenpei_3(nosFPGA,vertex,vertexname,nets):
	nc = len(vertexname)		# 节点个数
	N = 200		# 染色体群体规模
	maxiter = 500	# 最大循环次数
	SelfAdj = 0 	#SelfAdj = 1时为自适应
	randAccordDist = 0	# 在mutate的时候根据位置值分配对应的概率，此概率决定哪个城市参与mutate

	pc = 0.75		# 交叉概率
	pw = 0.25		# 变异概率
	# max_fitness = 0
	# max_b = np.zeros([0,nc])

	# 步骤1，产生N个染色体的初始群体,保存在pop里面
	pop = initPop(N,vertexname,nosFPGA)	#pop 遗传算法中的种群
	iter=0
	tstart=time.time()
	while iter<maxiter:
		iter = iter+1
		Sum_link=LinkCal(pop,nosFPGA,vertexname,nets);	#步骤2：计算每个染色体的路径长度
		ziyuan_list=ziyuanCal(pop,nosFPGA,vertex,vertexname);
  
		fitness=calFitness_3(Sum_link,ziyuan_list)			# 计算每个染色体的适应值	
		fitness = Restraint(pop,fitness,nosFPGA,vertex,vertexname)
		b,f,ind=findBest(pop,fitness)			# 找到在当前种群中，适应度最高的个体
		
		# if max_fitness<f:
		# 	max_fitness = f
		# 	max_b =  b[0,:]
		# b[0,:] = max_b
		# f = max_fitness
		
		
		# 步骤3：如果满足某些标准，算法停止 在此处没有设置条件
		if satified(fitness):
			break
		pop = chooseNewP(pop,fitness)				#      否则，选取出一个新的群体
		crossPop(pop,pc,fitness,SelfAdj)	#步骤4：交叉产生新染色体，得到新群体		
		mutPop(pop,pw,fitness,SelfAdj) 		#步骤5：基因变异	

		pop[0,:] = b[0,:]						# 保留上一代中适应值最高的染色体             
		onesum_link = linkCaloneGene(b[0,:],nosFPGA,vertexname,nets)
  
		if np.mod(iter,maxiter/10)==1:
			titer=time.time()
			print("iter = %d after running %d seconds with Link=%f" %(iter, int(titer-tstart),onesum_link))
   
	#输出最优染色体/路径
	print('nc=%d,N=%d, MAXITER=%d, pc=%f, pw=%f ' %(nc,N, maxiter,pc ,pw))
	print('最小连线为：%f,适应值为：%f,对应的分配方法为：' %(onesum_link,f))
	print(b[0,:])

	#计算运行时间
	tend=time.time()
	print('The program need :  %f seconds.' %(tend-tstart))

	#计算得到F
	F=[[] for i in range(nosFPGA)]
	for i in range(nc): 
		F[b[0][i]].append(vertex[vertexname[i]].id)
	return onesum_link,F

把结果文件打开

In [52]:
filename = "./result.res"
ft = open(filename,'w')
np.random.seed(np.random.randint(100))

第1种情况：

In [53]:
clearAssignInfo(vertex,vertexname,nets)
Std,F = GA_Fenpei_1(nosFPGA,vertex,vertexname)
print(F)
writeResult(F,vertexname)
ft.write(str(Std)+"\n")


iter = 1 after running 0 seconds with Std=21.358546
iter = 51 after running 0 seconds with Std=1.089725
iter = 101 after running 0 seconds with Std=0.433013
iter = 151 after running 1 seconds with Std=0.433013
iter = 201 after running 1 seconds with Std=0.433013
iter = 251 after running 2 seconds with Std=0.433013
iter = 301 after running 2 seconds with Std=0.433013
iter = 351 after running 3 seconds with Std=0.433013
iter = 401 after running 3 seconds with Std=0.433013
iter = 451 after running 4 seconds with Std=0.433013
nc=53,N=100, MAXITER=500, pc=0.500000, pw=0.500000 
最小方差为：0.433013,适应值为：2.309401,对应的分配方法为：
[1 1 0 0 2 3 2 3 2 0 3 3 3 2 0 3 3 3 2 2 2 2 2 2 2 2 3 3 2 2 3 0 2 0 3 1 3
 2 1 1 3 2 3 1 3 3 3 2 0 2 1 1 1]
The program need :  4.950493 seconds.
[[2, 3, 9, 14, 31, 33, 48], [0, 1, 35, 38, 39, 43, 50, 51, 52], [4, 6, 8, 13, 18, 19, 20, 21, 22, 23, 24, 25, 28, 29, 32, 37, 41, 47, 49], [5, 7, 10, 11, 12, 15, 16, 17, 26, 27, 30, 34, 36, 40, 42, 44, 45, 46]]


19

第二种情况

In [54]:
clearAssignInfo(vertex,vertexname,nets)
sum_link,F = GA_Fenpei_2(nosFPGA,vertex,vertexname,nets)
writeResult(F,vertexname)
ft.write(str(sum_link)+"\n")

iter = 1 after running 0 seconds with Sum=82.000000
iter = 71 after running 3 seconds with Sum=53.000000
iter = 141 after running 6 seconds with Sum=36.000000
iter = 211 after running 9 seconds with Sum=27.000000
iter = 281 after running 12 seconds with Sum=22.000000
iter = 351 after running 15 seconds with Sum=22.000000
iter = 421 after running 18 seconds with Sum=20.000000
iter = 491 after running 20 seconds with Sum=20.000000
iter = 561 after running 23 seconds with Sum=20.000000
iter = 631 after running 26 seconds with Sum=20.000000
nc=53,N=100, MAXITER=700, pc=0.750000, pw=0.250000 
最小连线为：20.000000,适应值为：0.050000,对应的分配方法为：
[3 0 3 3 3 3 1 1 3 3 1 1 3 3 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 3 3 1 1 1 0
 3 3 0 0 0 3 3 3 3 3 3 0 0 0 3 0]
The program need :  28.844455 seconds.


3

第三种情况

In [55]:
clearAssignInfo(vertex,vertexname,nets)
sum_link,F = GA_Fenpei_3(nosFPGA,vertex,vertexname,nets)
writeResult(F,vertexname)
ft.write(str(sum_link)+"\n")

iter = 1 after running 0 seconds with Link=97.000000
iter = 51 after running 5 seconds with Link=91.000000
iter = 101 after running 10 seconds with Link=90.000000
iter = 151 after running 15 seconds with Link=87.000000
iter = 201 after running 21 seconds with Link=97.000000
iter = 251 after running 27 seconds with Link=91.000000
iter = 301 after running 33 seconds with Link=87.000000
iter = 351 after running 38 seconds with Link=82.000000
iter = 401 after running 43 seconds with Link=87.000000
iter = 451 after running 48 seconds with Link=83.000000
nc=53,N=200, MAXITER=500, pc=0.750000, pw=0.250000 
最小连线为：74.000000,适应值为：0.001874,对应的分配方法为：
[3 3 2 1 2 3 2 1 3 1 3 2 1 2 1 2 2 1 1 0 3 1 2 1 1 1 3 1 1 0 0 1 2 2 0 1 1
 3 3 3 2 3 1 2 3 3 2 2 3 1 1 1 3]
The program need :  53.316828 seconds.


3

关闭文件

In [56]:
ft.close()

## 各种测试语句：

In [57]:
a1 = np.array([1,2,3])
a2 = np.array([1,2,3])
if (a1==a2).all():
    print(a1==a2)
if set(a1)==set(a2):
    print(" set(a1)==set(a2):")
    

[ True  True  True]
 set(a1)==set(a2):


In [58]:
print(fitness[:4])
fitness = np.cumsum(fitness)
print(fitness[:4])

NameError: name 'fitness' is not defined

In [ ]:
test = np.zeros([3,2],dtype=np.int8)
lst = [1,2]
test[1,:]=lst
print(test)
lst = []
print(test)

[[0 0]
 [1 2]
 [0 0]]
[[0 0]
 [1 2]
 [0 0]]


In [ ]:
# 读取节点文件
filename = "design.are"
vertex,vertexname = readNodeFile(filename)
# 读取线网文件
filename = "design.net"
nets = readNetFile(filename,vertex)

fenpei=[]
for i in range(N): #使用随机函数生成N个染色体            
    pop[i,:] =  [np.random.randint(0,nosFPGA) for j in range(53) ] # 给53个节点每个加上 0 1 2 3的随机数 ,randint 随机产生a-b之间的整数，包括a,numpy中randint不包括b。             

F = [[] for i in range(nosFPGA)]
gene = pop[1,:]
for i in range(nc): 
    F[gene[i]].append(vertex[vertexname[i]].ziyuan)
print(F)
print(sum(F[2]))


[[65, 65, 52, 6, 2, 6, 6, 1, 1, 1, 1], [53, 6, 6, 6, 3, 6, 6, 6, 7, 3, 1, 1, 1, 1, 1, 1, 1], [53, 64, 4, 1, 6, 3, 3, 4, 1, 1, 1, 1, 1, 1], [6, 6, 7, 3, 7, 4, 1, 1, 1, 1, 1]]
144


In [ ]:
print(F)

[[65, 65, 52, 6, 2, 6, 6, 1, 1, 1, 1], [53, 6, 6, 6, 3, 6, 6, 6, 7, 3, 1, 1, 1, 1, 1, 1, 1], [53, 64, 4, 1, 6, 3, 3, 4, 1, 1, 1, 1, 1, 1], [6, 6, 7, 3, 7, 4, 1, 1, 1, 1, 1]]


In [ ]:
np.std([276, 100, 24, 97])

92.74797841462637

In [ ]:
f=[2, 7, 24, 26, 36, 43, 45, 49, 52]
[vertex[vertexname[ind]].ziyuan for ind in f]

[65, 6, 6, 3, 1, 1, 1, 1, 1]

In [ ]:
sum([65, 6, 6, 3, 1, 1, 1, 1, 1])

85

In [ ]:
f=[5, 6, 10, 14, 15, 16, 19, 27, 30, 31, 32, 40, 41, 44, 46]
sum([vertex[vertexname[ind]].ziyuan for ind in f])

110

In [ ]:
f=[0, 1, 9, 11, 13, 17, 18, 25, 28, 29, 33, 34, 35, 37, 38, 42]
sum([vertex[vertexname[ind]].ziyuan for ind in f])

162

In [ ]:
f=[3, 4, 8, 12, 20, 21, 22, 23, 39, 47, 48, 50, 51]
sum([vertex[vertexname[ind]].ziyuan for ind in f])

140

In [ ]:
np.std([85,110,162,140])

29.22648627529488